# 全局特征实践课：颜色特征 · 灰度共生矩阵 · Gabor · HOG

**Global Features Lab — one image, one vector; checked against real labels**

本课使用两个真实图像数据集中的少量图片（见 `data/数据来源.txt`），每张图都有**真实类别标签**，因此"同类相近、异类远离"可以被精确统计：

| 数据 | 内容 | 用于 |
|---|---|---|
| `flowers/` | Oxford 17 Flowers 中 6 类 × 20 张彩色照片 | 颜色特征、HOG：以图搜图 |
| `textures/` | USC-SIPI 旋转纹理：13 种 Brodatz 纹理，各在 0°/30°/60°/90° **重新拍摄** | GLCM、Gabor：纹理分类与旋转 |

**本课任务：**

* **颜色特征、Gabor**：只需**调用现成函数**（`cv2.calcHist`、`cv2.compareHist`、`cv2.getGaborKernel`）完成特征提取与实验；
* **灰度共生矩阵（GLCM）、HOG**：做两种实现再对比——
  * **(A) 调用现成函数**：`skimage.feature.graycomatrix / graycoprops`、`skimage.feature.hog`；
  * **(B) 按课件步骤自己实现**：只用 NumPy 和基础运算，**不调用**对应的现成函数；
  * **(C) 对比**：结果是否一致、差在哪一步，以及在真实数据上的分类效果和不变性。

**要求**：下课前完成全部任务并提交本 Notebook（提交前 *Restart & Run All*，保留输出）。每个"问题"请**结合你自己运行得到的具体数字**回答。标 ★ 的为选做。

| 任务 | (A) 调用 | (B) 自己实现 | (C) 对比与实验 |
|---|---|---|---|
| 1 颜色特征 | `calcHist`、`compareHist` | — | — |
| 2 灰度共生矩阵 | `graycomatrix`、`graycoprops` | 共生矩阵 + 5 个统计量 | 逐元素一致性；纹理分类；旋转后"4 方向拼接"与"4 方向平均" |
| 3 Gabor | `getGaborKernel`、`filter2D` | — | — |
| 4 HOG | `skimage.feature.hog` | 梯度 → 9 bin 投票 → cell → block 归一化 → 串联 | 逐元素一致性；HOG 以图搜图，与颜色直方图对比；光照 / 翻转 / 旋转 |
| 5 总结 | | | |

## 0. 公共工具（已给出，直接运行）

约定：坐标 $(x, y)$，$x$ 向右、$y$ 向下；数组索引 `img[y, x]`。分类统一用**最近邻（1-NN）**：特征先按训练集做 z-score 标准化，再用欧氏距离找最近的训练样本。

In [ ]:
import os, sys, glob, time
import numpy as np
import cv2
import matplotlib
import matplotlib.pyplot as plt
import skimage
from skimage.feature import hog as sk_hog
try:
    from skimage.feature import graycomatrix, graycoprops          # skimage >= 0.19
except ImportError:
    from skimage.feature import greycomatrix as graycomatrix, greycoprops as graycoprops
print('Python', sys.version.split()[0], '| OpenCV', cv2.__version__, '| NumPy', np.__version__, '| skimage', skimage.__version__)

DATA = 'data'
for cand in ['data', os.path.join('..', 'data'), os.path.join('学生包', 'data')]:
    if os.path.isdir(cand):
        DATA = cand
        break
print('数据目录:', os.path.abspath(DATA))

# ---------- 花卉 ----------
FLOWER_CLASSES = ['sunflower', 'dandelion', 'bluebell', 'tigerlily', 'snowdrop', 'fritillary']
def load_flowers():
    ims, labels, names = [], [], []
    for c, nm in enumerate(FLOWER_CLASSES):
        for f in sorted(glob.glob(os.path.join(DATA, 'flowers', nm + '_*.jpg'))):
            ims.append(cv2.imread(f)); labels.append(c); names.append(os.path.basename(f))
    return ims, np.array(labels), names

def eval_retrieval(F, dist, labels, k=5):
    # 以图搜图：每张图作为查询，在其余 119 张中按距离排序；返回 (Top-1 准确率, Top-k 精度)
    n = len(F)
    D = np.array([[dist(F[i], F[j]) for j in range(n)] for i in range(n)], dtype=np.float64)
    np.fill_diagonal(D, np.inf)
    order = np.argsort(D, axis=1)
    return float((labels[order[:, 0]] == labels).mean()), float((labels[order[:, :k]] == labels[:, None]).mean())

# ---------- 纹理 ----------
TEX_NAMES = sorted(set(os.path.basename(f).split('_')[0] for f in glob.glob(os.path.join(DATA, 'textures', '*.png'))))
TEX_ANGLES = ['000', '030', '060', '090']
def load_texture_patches(angle):
    # 每张 512x512 纹理切成 4x4 = 16 个 128x128 块；每块做直方图均衡化（统一灰度分布，只留"排列方式"）
    X, y = [], []
    for c, nm in enumerate(TEX_NAMES):
        im = cv2.imread(os.path.join(DATA, 'textures', '%s_%s.png' % (nm, angle)), cv2.IMREAD_GRAYSCALE)
        for r in range(4):
            for cc in range(4):
                X.append(cv2.equalizeHist(np.ascontiguousarray(im[r * 128:(r + 1) * 128, cc * 128:(cc + 1) * 128]))); y.append(c)
    return X, np.array(y)
TEX = {a: load_texture_patches(a) for a in TEX_ANGLES}

def nn_predict(Ftr, ytr, Fte):
    Ftr = np.asarray(Ftr, float); Fte = np.asarray(Fte, float)
    mu, sd = Ftr.mean(0), Ftr.std(0) + 1e-9
    A, B = (Ftr - mu) / sd, (Fte - mu) / sd
    d = (B ** 2).sum(1)[:, None] + (A ** 2).sum(1)[None, :] - 2 * B @ A.T
    return ytr[np.argmin(d, axis=1)]

def eval_texture(feat_fn):
    # 训练：0° 的全部块；测试：30°/60°/90° 的全部块（同一纹理、重新拍摄的旋转版本）
    # 0° 一列是"留一法"：每个块在其余 0° 块中找最近邻（偏乐观，作为上限参考）
    F = {a: np.array([feat_fn(p) for p in TEX[a][0]]) for a in TEX_ANGLES}
    y = TEX['000'][1]
    res = {}
    mu, sd = F['000'].mean(0), F['000'].std(0) + 1e-9
    Z = (F['000'] - mu) / sd
    D = ((Z[:, None] - Z[None]) ** 2).sum(-1); np.fill_diagonal(D, np.inf)
    res['000'] = float((y[D.argmin(1)] == y).mean())
    for a in TEX_ANGLES[1:]:
        res[a] = float((nn_predict(F['000'], y, F[a]) == y).mean())
    return res

def print_tex(name, res):
    print('%-18s' % name, '  '.join('%s°: %.3f' % (a.lstrip('0') or '0', res[a]) for a in TEX_ANGLES))

def show(imgs, titles=None, size=2.6, cmap='gray'):
    fig, axes = plt.subplots(1, len(imgs), figsize=(size * len(imgs), size), squeeze=False)
    for i, im in enumerate(imgs):
        if im.ndim == 3: axes[0, i].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        else: axes[0, i].imshow(im, cmap=cmap)
        axes[0, i].set_axis_off()
        if titles: axes[0, i].set_title(titles[i], fontsize=9)
    plt.tight_layout(); plt.show()

FL_IMS, FL_Y, FL_NAMES = load_flowers()
show([FL_IMS[c * 20] for c in range(6)], FLOWER_CLASSES)
show([TEX['000'][0][c * 16] for c in range(0, 13, 2)], [TEX_NAMES[c] for c in range(0, 13, 2)])
show([cv2.imread(os.path.join(DATA, 'textures', 'brick_%s.png' % a), 0) for a in TEX_ANGLES], ['brick %s°' % a for a in TEX_ANGLES])

## 任务 1　颜色特征（调用）

1. 用 `cv2.cvtColor(im, cv2.COLOR_BGR2HSV)` 转到 HSV，用 `cv2.calcHist` 计算 **H×S×V = 8×4×4 = 128 维**直方图（OpenCV 中 H 的范围是 0–180），归一化使其和为 1；再计算 **RGB 8×8×8 = 512 维**直方图。
2. 用 `cv2.compareHist` 的四种度量——直方图交 `HISTCMP_INTERSECT`（距离取 1 − ∩）、卡方 `HISTCMP_CHISQR_ALT`、Bhattacharyya `HISTCMP_BHATTACHARYYA`，以及 L1（`np.abs(a-b).sum()`）——调用 `eval_retrieval` 做以图搜图，列出 Top-1 准确率和 Top-5 精度。
3. 颜色矩：HSV 三通道的均值、标准差、三阶矩（立方根）共 9 维，z-score 后用欧氏距离做同样的检索。
4. 不变性：对第一张向日葵做 水平翻转 / 旋转 90° / 缩小一半 / 亮度 ×1.3（`cv2.convertScaleAbs(im, alpha=1.3)`）/ 把图像切成 30×30 的块后随机打乱，分别计算与原图 HSV 直方图的直方图交。

In [ ]:
# TODO：hsv_hist、rgb_hist、四种距离的检索表、颜色矩检索、不变性实验

## 任务 2　灰度共生矩阵 GLCM（调用 + 自己实现 + 对比）

**(B) 自己实现**
1. `my_glcm(q, d, angle, L)`：`q` 是已量化到 0…L−1 的整数图像。方向按课件约定：0° 偏移 $(dx, dy) = (d, 0)$，45° 为 $(d, -d)$，90° 为 $(0, -d)$，135° 为 $(-d, -d)$。
   统计所有像素对 $\big(q[y, x],\ q[y+dy, x+dx]\big)$ 出现的次数，得到 $L\times L$ 矩阵 $M$；**对称化** $M \leftarrow M + M^T$；**归一化**使元素和为 1。（提示：把两个灰度编码成一个整数 `i * L + j`，再用 `np.bincount`。）
2. `my_props(P)`：返回 5 个统计量（$i, j$ 为行列下标）：
   对比度 $\sum P_{ij}(i-j)^2$；能量（角二阶矩）ASM $\sum P_{ij}^2$；熵 $-\sum P_{ij}\log_2 P_{ij}$（只对 $P_{ij}>0$ 求和）；
   逆差矩 $\sum \frac{P_{ij}}{1+(i-j)^2}$；相关性 $\sum \frac{(i-\mu_i)(j-\mu_j)P_{ij}}{\sigma_i\sigma_j}$，其中 $\mu_i=\sum_{ij} iP_{ij}$，$\sigma_i^2=\sum_{ij}(i-\mu_i)^2P_{ij}$，$\mu_j,\sigma_j$ 同理。

**(A) 调用**：`graycomatrix(q, [d], [angle_rad], levels=L, symmetric=True, normed=True)` 与 `graycoprops(G, 'contrast' / 'ASM' / 'homogeneity' / 'correlation')`（熵没有现成属性，用 NumPy 由 G 计算）。

**(C) 对比**
1. 在一个纹理块（量化到 L = 16）上比较两种实现：**先取 d = 1、用相同角度直接比较，看哪些方向对不上；再根据 skimage 的偏移约定（行偏移 `round(d·sinθ)`、列偏移 `round(d·cosθ)`，y 轴向下）找出与课件方向的对应关系，重新比较矩阵和统计量**；最后看看 d = 2、4 时对角方向还一致吗；
2. 用你的实现组成两种特征：**4 方向拼接**（5 统计量 × 3 距离 × 4 方向 = 60 维）和 **4 方向平均**（对方向取平均，15 维），分别用 `eval_texture` 测试 0° 训练、30°/60°/90° 测试的准确率；
3. 对照：只用 32 bin 灰度直方图（`cv2.calcHist`）做同样的测试。

In [ ]:
ANGLES = {0: (1, 0), 45: (1, -1), 90: (0, -1), 135: (-1, -1)}   # 单位偏移 (dx, dy)，乘以 d

def quantize(img, L=16):
    return (img.astype(np.int32) * L // 256).astype(np.uint8)

def my_glcm(q, d, angle, L=16):
    # TODO (B)：返回 L x L 的 float64 矩阵（对称、归一化）
    raise NotImplementedError

def my_props(P):
    # TODO (B)：返回 dict，键为 'contrast', 'ASM', 'entropy', 'homogeneity', 'correlation'
    raise NotImplementedError

In [ ]:
# TODO (C) 1：与 skimage 逐元素对比

In [ ]:
# TODO (C) 2、3：4 方向拼接 vs 4 方向平均 vs 灰度直方图 的纹理分类

**问题 1**：(a) 直接按相同角度比较时，哪些方向对不上？原因是什么？按正确对应关系比较后最大差多少？d = 2 时对角方向为什么又不一致？"对称化"和"归一化"各起什么作用？
(b) 0° 时"4 方向拼接"与"4 方向平均"谁更准？旋转到 30°/60°/90° 后呢？为什么拼接版对旋转敏感，平均为什么能缓解？90° 旋转恰好把 0°↔90°、45°↔135° 对调，拼接版为什么仍然下降？
(c) 灰度直方图的准确率是多少？为什么在这份数据上几乎等于随机猜测（提示：`load_texture_patches` 对每个块做了什么）？

> 你的回答：

## 任务 3　Gabor 特征（调用）

1. 用 `cv2.getGaborKernel((31, 31), sigma, theta, lambd, gamma, psi)` 构造滤波器组：波长 λ ∈ {4, 6, 9, 13}，方向 θ = 0°, 30°, …, 150°（6 个），`sigma = 0.56 * λ`，`gamma = 0.5`；
   每组 (λ, θ) 取 `psi = 0`（偶对称）和 `psi = π/2`（奇对称）两个核，各自减去均值、除以绝对值之和。
2. 对纹理块（转成 float32）分别用两个核 `cv2.filter2D`，响应模长 $\sqrt{e^2 + o^2}$，取其**均值和标准差** → 4 × 6 × 2 = 48 维；
3. 旋转不变版本：对每个尺度把 6 个方向的均值、标准差再分别取平均 → 4 × 2 × 2 = 16 维；
4. 用 `eval_texture` 测试两种 Gabor 特征，并画出 λ = 6 时 6 个方向核的形状。

In [ ]:
# TODO：Gabor 滤波器组、48 维 / 16 维（旋转不变）特征、纹理分类

## 任务 4　HOG（调用 + 自己实现 + 对比）

HOG 描述的是"哪里有什么方向的边"，也就是形状。这里把它用在花卉图像上：每张图转成灰度并缩放到 **64×64**，与任务 1 的颜色直方图比较——颜色和形状，谁更能区分这 6 种花？

**(B) 自己实现** `my_hog(img, interp=True)`，按课件 5 步：
1. **梯度**：中心差分 $[-1, 0, 1]$，$g_x[y,x] = I[y,x+1]-I[y,x-1]$，$g_y[y,x] = I[y+1,x]-I[y-1,x]$（图像最外一圈梯度记 0，不做平滑）；幅值 $|g|$，**无符号**方向 $\theta \in [0°, 180°)$；
2. **方向投票**：9 个 bin，每 20° 一个。
   * `interp=True`（课件做法，双线性）：bin 中心为 0°, 20°, …, 160°（160° 与 0° 相邻），把 $|g|$ 按距离分给相邻两个 bin，例如 72.6° → 60° 分 7.4/20、80° 分 12.6/20；
   * `interp=False`（硬分配）：$\lfloor\theta/20\rfloor$ 号 bin 得到全部 $|g|$；
3. **Cell**：8×8 像素一个 cell，每个 cell 一个 9 维直方图（64×64 图像 → 8×8 个 cell）；
4. **Block 归一化**：2×2 个 cell 组成一个 block（36 维），步长 1 个 cell；L2-Hys：$v \leftarrow v / \sqrt{\lVert v\rVert^2 + 10^{-10}}$，截断到 0.2，再按同样方式归一化一次；
5. **串联**：7 × 7 个 block 按"行优先"依次拼接 → 7 × 7 × 36 = **1764 维**。

**(A) 调用**：`sk_hog(img, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2), block_norm='L2-Hys')`。

**(C) 对比**
1. 在第一张向日葵上比较：`my_hog(interp=False)` 与 skimage 的最大差；`my_hog(interp=True)` 与 skimage 的相关系数；
2. 以图搜图（`eval_retrieval`，欧氏距离）：自己的 HOG（双线性）、自己的 HOG（硬分配）、skimage HOG、原始灰度像素（64×64 = 4096 维），并与任务 1 中 HSV 直方图 + Bhattacharyya 的结果对比；再分别列出 HOG 与 HSV 直方图**每一类**的 Top-1 准确率；
3. 不变性：把每张图做 亮度 ×0.5 / 水平翻转 / 上下颠倒 / 旋转 90° / 旋转 20° 后作为查询，在 120 张原图中检索，统计**排第一的恰好是它自己的原图**的比例；再验证：亮度 ×1.5（用 float，不截断）前后 `my_hog` 是否完全相同。

In [ ]:
def my_hog(img, interp=True):
    # TODO (B)：返回 float64 向量（64x64 输入时为 1764 维）
    raise NotImplementedError

In [ ]:
def flower_gray(im, size=64):
    return cv2.resize(cv2.cvtColor(im, cv2.COLOR_BGR2GRAY), (size, size), interpolation=cv2.INTER_AREA)
FL_GRAY = [flower_gray(im) for im in FL_IMS]
# TODO (C) 1：与 skimage 对比（并画出 skimage 的 HOG 可视化）

In [ ]:
# TODO (C) 2、3：HOG 以图搜图 vs 颜色直方图（含每类准确率）；不变性实验；亮度 x1.5 验证

**问题 2**：(a) 硬分配版与 skimage 是否一致？双线性版与 skimage 相关系数多少？两者差在哪一步（bin 中心与投票方式）？检索准确率差多少？
(b) HOG、原始像素、HSV 颜色直方图的 Top-1 各是多少？对比每一类：哪些花 HOG 更好、哪些花颜色更好？结合花的形状和颜色解释（提示：向日葵和蒲公英颜色相近）。
(c) 亮度 ×0.5、水平翻转、上下颠倒、旋转 90°、旋转 20° 后找回自己的比例各是多少？哪一步设计让 HOG 对亮度不敏感？为什么它对翻转和旋转都很敏感？这和任务 1 中颜色直方图的不变性正好相反吗？

> 你的回答：

## 任务 5　总结

1. 结合本课实验结果，填写下表（✓ / △ / ✗，并在括号中写出支撑它的一个数字）：

| 特征 | 旋转 | 亮度变化 | 保留空间布局 | 最适合的任务 |
|---|---|---|---|---|
| 颜色直方图 | | | | |
| 颜色矩 | | | | |
| GLCM（4 方向拼接 / 平均） | | | | |
| Gabor（48 维 / 旋转不变） | | | | |
| HOG | | | | |

2. GLCM 和 HOG：你自己的实现与库函数的主要差别分别是什么？各用一个数字说明。

> 你的回答：